# MedicalReasoning — *"what drugs treat &lt;disease&gt;?"*

A small, grounded pipeline over RTX-KG2. The LLM handles language; DuckDB and Soufflé stay
deterministic so entities and rules are always tied to the real graph.

```
question ─▶ [LLM] parse disease ─▶ [DuckDB] resolve to real KG2 id ─▶ pick one
                                                                          │
   named drug list ◀─ [DuckDB] name results ◀─ [Soufflé] scan edges ◀─ [LLM] write .dl
```

Scope for now: one predicate, `biolink:treats`. Brute-force — Soufflé scans the raw 38 GB edges
file directly each run (no preprocessing, no index).


## Setup

In [6]:
import os
import json
import duckdb
import pandas as pd
from dotenv import load_dotenv
import anthropic

pd.set_option("display.max_colwidth", 80)

NODES_PARQUET = "nodes_index.parquet"            # lean node lookup index (built below)
DISEASE_CATEGORIES = [                            # what counts as a disease/phenotype in KG2
    "biolink:Disease", "biolink:DiseaseOrPhenotypicFeature",
    "biolink:PhenotypicFeature", "biolink:PathologicalProcess",
]

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")

load_dotenv()                                     # reads .env -> APIKEY
client = anthropic.Anthropic(api_key=os.environ["APIKEY"])
MODEL  = "claude-sonnet-4-5"

def llm(system, user, max_tokens=1024):
    msg = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        system=system, messages=[{"role": "user", "content": user}],
    )
    return msg.content[0].text

## Step 1 & 2 — parse the disease (LLM), resolve it to real KG2 IDs (DuckDB)

In [7]:
def parse_disease(query):
    """LLM: pull the single disease/condition out of the user's question."""
    system = ('Extract the single disease or condition the question is about. '
              'Reply with ONLY JSON of the form {"disease": "<common medical name>"}.')
    raw = llm(system, query, max_tokens=100).strip().removeprefix("```json").strip("`").strip()
    return json.loads(raw)["disease"]

def resolve_disease(term, limit=20):
    """DuckDB: rank real disease nodes whose name/synonym matches `term`."""
    t = term.strip().lower()
    cats = "(" + ",".join(f"'{c}'" for c in DISEASE_CATEGORIES) + ")"
    q = f"""
        SELECT id, name, category,
               CASE WHEN name_lc = ? THEN 0 WHEN name_lc LIKE ? THEN 1
                    WHEN name_lc LIKE ? THEN 2 ELSE 3 END AS rank
        FROM '{NODES_PARQUET}'
        WHERE category IN {cats}
          AND (name_lc LIKE ? OR lower(coalesce(synonyms, '')) LIKE ?)
        ORDER BY rank, length(name)
        LIMIT {limit}
    """
    contains, prefix = f"%{t}%", f"{t}%"
    return con.execute(q, [t, prefix, contains, contains, contains]).df()[["id", "name", "category"]]

## Step 3 — disambiguate (user picks)

In [8]:
def pick_from(df, what="match"):
    df = df.reset_index(drop=True)
    if len(df) == 0:
        print("No candidates found."); return None
    if len(df) == 1:
        print(f"Single {what}: {df.iloc[0]['name']}  [{df.iloc[0]['id']}]"); return df.iloc[0]
    print(f"Multiple candidates for the {what} — pick one:\n")
    for i, row in df.iterrows():
        print(f"  [{i:>2}] {row['name'][:55]:55s} {row['category']:35s} {row['id']}")
    while True:
        sel = input("\nEnter number (blank to cancel): ").strip()
        if sel == "":
            return None
        if sel.isdigit() and int(sel) in df.index:
            return df.loc[int(sel)]
        print("Invalid selection, try again.")

## Step 4 — write the Soufflé `.dl` (LLM)

Constrained generation: the LLM gets the fact schema and the real predicate vocabulary, and must
ground the entity with the exact CURIE we resolved. No preprocessing — the program reads the
38 GB edges file **directly**, selecting only the 4 columns it needs (`columns="16:0:17:3"` =
subject, core_predicate, object, negated), so the giant description field never enters memory.

In [9]:
EDGES_TSV = "rtx-kg2_edges_2.8.1.tsv"
KG2_PREDICATES = [  # treatment-relevant slice of the 61 real KG2 core_predicate values
    "biolink:treats", "biolink:affects", "biolink:causes", "biolink:predisposes",
    "biolink:prevents", "biolink:contraindicated_for", "biolink:gene_associated_with_condition",
    "biolink:has_phenotype", "biolink:biomarker_for", "biolink:related_to",
]

# Every program starts with this header: one relation read straight from the edges file.
EDGE_HEADER = (
    ".decl edge(subj:symbol, pred:symbol, obj:symbol, neg:symbol)\n"
    f'.input edge(IO=file, filename="{EDGES_TSV}", columns="16:0:17:3")'
)

DL_SYSTEM = (
    "You write Soufflé Datalog that queries the RTX-KG2 biomedical knowledge graph.\n"
    "Every program MUST begin with exactly these two lines:\n"
    + EDGE_HEADER + "\n\n"
    "edge(subj, pred, obj, neg) is one KG edge: subj-[pred]->obj are CURIEs; neg is \"True\"/"
    "\"False\" (use only neg=\"False\" facts). pred is a biolink predicate such as: "
    + ", ".join(KG2_PREDICATES) + ".\n"
    "Direction: biolink:treats runs subject=drug -> object=disease.\n\n"
    "Requirements:\n"
    "- Put the final answer in a relation named EXACTLY `result`, and end with `.output result`.\n"
    "- Ground every entity with the EXACT CURIE the user gives; never invent ids.\n"
    "- Output ONLY the Soufflé program (no prose, no markdown). lowercase names, avoid "
    'built-ins like "cat".'
)

def generate_dl_llm(query, disease_id, disease_name, out_path="query.dl"):
    """LLM writes the .dl for `query`, grounded with the resolved id."""
    user = (f"Question: {query}\n"
            f'Grounded entity: {disease_name} = "{disease_id}"\n'
            f"Write the Soufflé program.")
    dl = llm(DL_SYSTEM, user, max_tokens=600).strip().removeprefix("```souffle").strip("`").strip()
    with open(out_path, "w") as f:
        f.write(dl + "\n")
    print(f"Wrote {out_path}\n\n{dl}")
    return out_path

## Step 5 — run it over the edges (brute force) and name the results

No index, no preprocessing: Soufflé streams the whole 38 GB edges file into memory once
(~minutes, a few GB RAM) and evaluates the rule. We then map the resulting CURIEs back to
human-readable names via the node index.

In [10]:
import subprocess

def run_dl(dl_path="query.dl", out_dir="souffle_out"):
    """Run Soufflé on the .dl (brute-force scan of the edges file), then name the result ids."""
    os.makedirs(out_dir, exist_ok=True)
    print(f"Running Soufflé over {EDGES_TSV} (full scan, a few minutes)...")
    subprocess.run(["souffle", dl_path, "-F", ".", "-D", out_dir], check=True)

    out = os.path.join(out_dir, "result.csv")          # .output result -> result.csv (tab-sep)
    rows = pd.read_csv(out, sep="\t", header=None, dtype=str) if os.path.getsize(out) else pd.DataFrame()
    if rows.empty:
        print("No results."); return rows
    ids = rows[0].tolist()                              # first column = the answer CURIEs
    idlist = "(" + ",".join(f"'{i}'" for i in ids) + ")"
    named = con.execute(
        f"SELECT id, name, category FROM '{NODES_PARQUET}' WHERE id IN {idlist}"
    ).df()
    print(f"{len(named)} result(s):")
    return named

## Run the pipeline

In [12]:
query = input("Ask a question (e.g. 'what drugs treat parkinson disease?'): ").strip()

disease = parse_disease(query)                      # 1. LLM
print(f"Disease parsed from question: {disease!r}\n")

candidates = resolve_disease(disease)               # 2. DuckDB -> real KG2 ids
chosen = pick_from(candidates, what="disease")      # 3. user picks

if chosen is not None:
    print(f"\nResolved: {chosen['name']} -> {chosen['id']}\n")
    generate_dl_llm(query, chosen["id"], chosen["name"])   # 4. LLM writes query.dl
    answer = run_dl("query.dl")                             # 5. Soufflé scans edges -> named results
    display(answer)

Disease parsed from question: "Parkinson's disease"

Multiple candidates for the disease — pick one:

  [ 0] Parkinson's Disease                                     biolink:Disease                     NCIT:C26845
  [ 1] Parkinson's disease                                     biolink:Disease                     DOID:14330
  [ 2] Parkinson's disease                                     biolink:Disease                     ICD9:332
  [ 3] Parkinson's disease                                     biolink:Disease                     LOINC:MTHU020807
  [ 4] Parkinson's disease                                     biolink:Disease                     LOINC:LP75351-4
  [ 5] Parkinson's disease                                     biolink:Disease                     LOINC:LA27533-1
  [ 6] Parkinson's disease 4                                   biolink:Disease                     DOID:0060895
  [ 7] Parkinson's disease 6                                   biolink:Disease                     DOID:0060369

,id,name,category
0,DrugCentral:144,amantidine,biolink:ChemicalEntity
1,DrugCentral:1567,DOPA,biolink:ChemicalEntity
2,DrugCentral:374,biperiden HCl,biolink:ChemicalEntity
3,DrugCentral:971,threo-Dopaserine,biolink:ChemicalEntity
4,DrugCentral:2276,kemadrine,biolink:ChemicalEntity
5,DrugCentral:2429,selegiline hydrochloride,biolink:ChemicalEntity
6,DrugCentral:4882,KW-6002,biolink:ChemicalEntity
7,DrugCentral:5143,BIA 9-1067,biolink:ChemicalEntity
8,DrugCentral:228,apomorphine HCl,biolink:ChemicalEntity
9,DrugCentral:2233,pramipexole dihydrochloride,biolink:ChemicalEntity
